In [1]:
# 라이브러리 설치 필요한 경우
#!pip install pandas numpy matplotlib seaborn folium ipywidgets scikit-learn

In [2]:
# 필요 라이브러리
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# heatmap, barplot처럼 보기 좋은 그래프 그릴 떄 사용
import seaborn as sns
# 위도/경도 데이터를 지도 위에 표시할 때 사용
import folium
# folium 지도 위에 데이터 밀집 정도를 색으로 표현
from folium.plugins import HeatMap

In [3]:
plt.style.use('ggplot')

In [4]:
# 한글 폰트 설정(예외처리: 폰트 없거나 다른 OS에서 실행해도 오류X)
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print(e)

### 데이터 불러오기

In [5]:
df = pd.read_csv('jeju_bus.csv')

### 데이터 기본 구조 확인

In [6]:
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


### 데이터 크기 확인 (shape)

In [7]:
df.shape

(210457, 14)

### 컬럼 타입과 결측 여부 확인 (info_

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  float64
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 22.5 MB


### 숫자형 컬럼의 기초 통계 확인 (describe)

In [9]:
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000


### 결측치 확인
- isnull().sum(): 열별 결측 개수
- isnull().mean(): 열별 결측 비율

In [10]:
missing_count = df.isnull().sum()
missing_ratio = df.isnull().mean()

missing_df = pd.DataFrame({
    'missing_count':missing_count,
    'missing_ratio':missing_ratio
})

missing_df

,missing_count,missing_ratio
id,0,0.0
date,0,0.0
route_id,0,0.0
vh_id,0,0.0
route_nm,0,0.0
now_latitude,0,0.0
now_longitude,0,0.0
now_station,0,0.0
now_arrive_time,0,0.0
distance,0,0.0


### tl

### now_arrive_time → 숫자형 now_hour

In [11]:
df['now_hour'] = df['now_arrive_time'].str.replace('시','',regex=False).astype(int)

In [12]:
df[['now_arrive_time', 'now_hour']].head()

,now_arrive_time,now_hour
0,06시,6
1,06시,6
2,06시,6
3,06시,6
4,07시,7


### data → 날짜형 변환 및 파생변수 생성

In [13]:
df['date'] = pd.to_datetime(df['date'])

In [14]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

In [15]:
df[['date', 'year', 'month', 'day', 'dayofweek']].head()

,date,year,month,day,dayofweek
0,2019-10-15,2019,10,15,1
1,2019-10-15,2019,10,15,1
2,2019-10-15,2019,10,15,1
3,2019-10-15,2019,10,15,1
4,2019-10-15,2019,10,15,1


### target 기초 통계

In [16]:
df['next_arrive_time'].describe()

count    210457.000000
mean         85.380824
std          85.051170
min           6.000000
25%          44.000000
50%          66.000000
75%         102.000000
max        2996.000000
Name: next_arrive_time, dtype: float64

### 히스토그램 (값이 어디에 몰려 있는가)

plt.figure(figsize=(15, 8))
plt.hist(df['next_arrive_time'], bins = 50)
plt.title('Distribution of next_arrive_time')
plt.xlabel('next_arrive_time')
plt.ylabel('count')
plt.show()

### 박스 플롯 (이상치 후보 확인)

plt.figure(figsize=(20, 8))
plt.boxplot(df['next_arrive_time'], vert = False)
plt.title('Boxplot of next_arrive_time')
plt.xlabel('next_arrive_time')
plt.show()

### 이상치 확인

In [19]:
time_quantile = df['next_arrive_time'].quantile([0.90, 0.95, 0.99])
time_quantile

0.90    154.0
0.95    194.0
0.99    340.0
Name: next_arrive_time, dtype: float64

In [20]:
q90 = df['next_arrive_time'].quantile(0.90)

df = df[df['next_arrive_time'] <= q90]

In [21]:
over_300_count = (df['next_arrive_time'] > 300).sum()
over_600_count = (df['next_arrive_time'] > 600).sum()
over_1000_count = (df['next_arrive_time'] > 1000).sum()

print('300 초과 :', over_300_count)
print('600 초과 :', over_600_count)
print('1000 초과 :', over_1000_count)

300 초과 : 0
600 초과 : 0
1000 초과 : 0


q90 = df['next_arrive_time'].quantile(0.90)

df_under_q90 = df[df['next_arrive_time'] <= q90]

plt.figure(figsize=(15, 8))
plt.scatter(
    df_under_q90['distance'],
    df_under_q90['next_arrive_time'],
    alpha = 0.2,
    s = 5
)

plt.title('Distance vs Next Arrive Time under 90% qyantule')
plt.xlabel('distance')
plt.ylabel('next_arrive_time')
plt.show()

### 상관관계 heatmap

In [23]:
corr_cols = [
    'distance',
    'next_arrive_time',
    'now_hour'    
]

corr_data = df[corr_cols]
corr_matrix = corr_data.corr()
corr_matrix

,distance,next_arrive_time,now_hour
distance,1.000000,0.366562,-0.001620
next_arrive_time,0.366562,1.000000,-0.011816
now_hour,-0.001620,-0.011816,1.000000


plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot = True, cmap = 'coolwarm')
plt.title('correlation Heatmap')
plt.show()

### 시간대별 데이터 개수

In [25]:
hour_count = df.groupby('now_hour')['next_arrive_time'].count()
hour_count

now_hour
0         5
5       424
6      7668
7     12292
8     12311
9     12473
10    11823
11    11358
12    12226
13    11992
14    11402
15    11588
16    12101
17    11565
18    11288
19    11786
20    10745
21    10266
22     6022
23      445
Name: next_arrive_time, dtype: int64

In [26]:
hour_mean_time = df.groupby('now_hour')['next_arrive_time'].mean()
hour_mean_time

now_hour
0     56.000000
5     65.433962
6     68.364632
7     68.445330
8     68.512387
9     67.555360
10    67.792777
11    68.443828
12    68.169720
13    68.786524
14    68.083933
15    68.197100
16    69.093298
17    70.043926
18    70.585135
19    67.975564
20    66.206794
21    65.304500
22    64.802723
23    61.146067
Name: next_arrive_time, dtype: float64

In [27]:
hour_summary = df.groupby('now_hour')['next_arrive_time'].agg(
    ['count', 'mean', 'median']
)
hour_summary

,count,mean,median
now_hour,,,
0,5,56.000000,36.0
5,424,65.433962,56.0
6,7668,68.364632,60.0
7,12292,68.445330,60.0
8,12311,68.512387,62.0
9,12473,67.555360,60.0
10,11823,67.792777,60.0
11,11358,68.443828,62.0
12,12226,68.169720,62.0


hour_summary['mean'].plot(kind='bar',figsize = (10,4))
plt.title('Average next_arrive_time by Hour')
plt.xlabel('Hour')
plt.ylabel('Average next_arrive_time')
plt.show()

### 요일별 도착 시간

In [29]:
day_count = df.groupby('dayofweek')['next_arrive_time'].count()
day_count

dayofweek
0    27331
1    28166
2    27736
3    27933
4    26875
5    26056
6    25683
Name: next_arrive_time, dtype: int64

In [30]:
day_mean_time = df.groupby('dayofweek')['next_arrive_time'].mean()
day_mean_time

dayofweek
0    68.573415
1    68.178087
2    68.736408
3    68.663122
4    68.578456
5    67.062941
6    66.869836
Name: next_arrive_time, dtype: float64

In [31]:
day_summary = df.groupby('dayofweek')['next_arrive_time'].agg(
    ['count', 'mean', 'median']
)
day_summary

,count,mean,median
dayofweek,,,
0,27331,68.573415,62.0
1,28166,68.178087,60.0
2,27736,68.736408,62.0
3,27933,68.663122,62.0
4,26875,68.578456,62.0
5,26056,67.062941,60.0
6,25683,66.869836,58.0


day_summary['mean'].plot(kind='bar',figsize = (8,4))
plt.title('Average next_arrive_time by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average next_arrive_time')
plt.show()

### 노선별 도착 시간 (개수 → 평균 → 정리)

In [33]:
route_count = df.groupby('route_nm')['next_arrive_time'].count()
route_count

route_nm
201-11     7427
201-12    11292
201-13     2467
201-14    12478
201-15     2129
201-16    14723
201-17     4791
201-18     2433
201-21     2521
201-22     7439
201-24     2262
201-26     2574
201-27     5489
281-1     17274
281-2     23954
360-1     19001
360-12     6780
360-2      6562
360-7      1816
365-21     9480
365-22    26888
Name: next_arrive_time, dtype: int64

In [34]:
route_mean_time = df.groupby('route_nm')['next_arrive_time'].mean()
route_mean_time

route_nm
201-11    60.044836
201-12    59.173574
201-13    59.070126
201-14    60.195865
201-15    60.119305
201-16    60.097806
201-17    60.271342
201-18    58.355117
201-21    57.586672
201-22    58.681006
201-24    57.678603
201-26    58.568765
201-27    60.915832
281-1     68.502258
281-2     68.829924
360-1     77.595179
360-12    77.867257
360-2     78.511429
360-7     79.514317
365-21    80.708228
365-22    74.972144
Name: next_arrive_time, dtype: float64

In [35]:
route_summary = df.groupby('route_nm')['next_arrive_time'].agg(
    ['count', 'mean', 'median']
)
route_summary = route_summary.sort_values('mean', ascending = False)
route_summary

,count,mean,median
route_nm,,,
365-21,9480,80.708228,76.0
360-7,1816,79.514317,74.0
360-2,6562,78.511429,76.0
360-12,6780,77.867257,74.0
360-1,19001,77.595179,72.0
365-22,26888,74.972144,70.0
281-2,23954,68.829924,66.0
281-1,17274,68.502258,64.0
201-27,5489,60.915832,52.0


hour_summary['mean'].plot(kind='bar',figsize = (12,4))
plt.title('Average next_arrive_time by Route')
plt.xlabel('route_nm')
plt.ylabel('Average next_arrive_time')
plt.xticks(rotation=45)
plt.show()

### 정류장 구간(station_segment)별 도착 시간

In [37]:
df['station_segment'] = (
    df['now_station'].astype(str)
    + " → "
    +df['next_station'].astype(str)
)

In [38]:
df[['now_station', 'next_station', 'station_segment']]

,now_station,next_station,station_segment
0,제주대학교입구,제대마을,제주대학교입구 → 제대마을
1,제대마을,제대아파트,제대마을 → 제대아파트
2,제대아파트,제주대학교,제대아파트 → 제주대학교
3,남국원(아라방면),제주여자중고등학교(아라방면),남국원(아라방면) → 제주여자중고등학교(아라방면)
4,도호동,은남동,도호동 → 은남동
...,...,...,...
210452,비석거리,삼아아파트,비석거리 → 삼아아파트
210453,동문로터리,매일올레시장 7번입구,동문로터리 → 매일올레시장 7번입구
210454,서귀포시 구 버스터미널,아랑조을거리 입구,서귀포시 구 버스터미널 → 아랑조을거리 입구
210455,아랑조을거리 입구,평생학습관,아랑조을거리 입구 → 평생학습관


In [39]:
segment_count = df.groupby('station_segment')['next_arrive_time'].count()
segment_count.sort_values(ascending = False).head(10)

station_segment
남국원(아라방면) → 아라초등학교                    1902
은남동 → 도호동                             1898
아라주공아파트 → 인다마을                        1849
인다마을 → 제주대학교병원                        1777
제주여자중고등학교(아라방면) → 남국원(아라방면)           1775
아라초등학교 → 아라주공아파트                      1751
제주시청(아라방면) → 고산동산(아라방면)               1727
제주중앙여자고등학교(아라방면) → 제주여자중고등학교(아라방면)    1673
고산동산(아라방면) → 제주지방법원(아라방면)             1633
도호동 → 은남동                             1605
Name: next_arrive_time, dtype: int64

In [40]:
segment_mean_time = df.groupby('station_segment')['next_arrive_time'].mean()
segment_mean_time.sort_values(ascending = False).head(10)

station_segment
제주여자중고등학교(아라방면) → 제주중앙여자고등학교(아라방면)    149.756098
천수동 → 고으니모르 국립제주박물관                   149.615385
입석동 → 한라산 둘레길                         148.611111
제주도청 신제주로터리 → 제주국제공항(구제주방면)           148.000000
종달초등학교 → 시흥리                          144.574713
시흥리 → 종달초등학교                          142.542373
제주국제공항(신제주방면) → 다호마을                  142.098361
동광양 → 문예회관                            141.640000
행원농공단지 → 한동리서동                        140.800000
탐라장애인 종합복지관 → 동광양                     140.126582
Name: next_arrive_time, dtype: float64

In [41]:
segment_summary = df.groupby('station_segment')['next_arrive_time'].agg(
    ['count', 'mean', 'median']
)

segment_summary = segment_summary[segment_summary['count'] >= 100]
segment_summary = segment_summary.sort_values('mean', ascending = False)
segment_summary

,count,mean,median
station_segment,,,
종달초등학교 → 시흥리,174,144.574713,146.0
제주국제공항(신제주방면) → 다호마을,122,142.098361,146.0
동광양 → 문예회관,100,141.640000,148.0
한동리서동 → 행원농공단지,187,137.668449,138.0
하례환승정류장(하례리입구) → 입석동,183,137.431694,138.0
...,...,...,...
국립제주박물관 → 우당도서관,213,22.535211,22.0
성산리 → 성산리 취락구조,175,20.640000,18.0
평대초등학교 → 평대리서동,267,20.520599,20.0


### 위치 기반 시각화

In [42]:
up = (33.506286, 126.430312)# 제주공항
right = (33.493521, 126.895326)# 성산일충봉
center = (33.379724, 126.545315)# 한라산
down = (33.246742, 126.562387)# 서귀포시

In [43]:
station_location_df = df.copy()

jeju_map = folium.Map(
    location = [33.38, 126.55],
    zoom_start = 10
)

for _, row in station_location_df.iterrows():
    folium.CircleMarker(
        location = [row['now_latitude'], row['now_longitude']],
        radius = 2,
        fill = True,
        fill_opacity = 0.5,
        popup = row['now_station']
    ).add_to(jeju_map)

folium.Marker(
    location=[up[0], up[1]],
    popup = '제주공항',
    tooltip = '제주공항',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_map)

folium.Marker(
    location=[right[0], right[1]],
    popup = '성산일출봉 방명',
    tooltip = '성산일출봉 방명',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_map)

folium.Marker(
    location=[center[0], center[1]],
    popup = '한라상 / 중산갓',
    tooltip = '한라상 / 중산갓',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_map)

folium.Marker(
    location=[down[0], down[1]],
    popup = '서귀포 방면',
    tooltip = '서귀포 방면',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_map)

# 필요하면 저장
#jeju_map.save('jeju_bus_stop.map.html')

jeju_map

jeju_heatmap = folium.Map(
    location = [33.38, 126.55],
    zoom_start = 10
)

heat_data = station_location_df[['now_latitude', 'now_longitude']].values.tolist()

HeatMap(
    heat_data,
    radius = 15,
    blur = 20
).add_to(jeju_heatmap)

folium.Marker(
    location=[up[0], up[1]],
    popup = '제주공항',
    tooltip = '제주공항',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_heatmap)

folium.Marker(
    location=[right[0], right[1]],
    popup = '성산일출봉 방명',
    tooltip = '성산일출봉 방명',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_heatmap)

folium.Marker(
    location=[center[0], center[1]],
    popup = '한라상 / 중산갓',
    tooltip = '한라상 / 중산갓',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_heatmap)

folium.Marker(
    location=[down[0], down[1]],
    popup = '서귀포 방면',
    tooltip = '서귀포 방면',
    icon = folium.Icon(color = 'blue')
).add_to(jeju_heatmap)

# 필요하면 저장
#jeju_heatmap.save('jeju_bus_stop.heatmap.html')

jeju_heatmap

In [46]:
analysis_df = df.copy()

### 도착 시간이 긴 데이터 확인

In [47]:
time_q99 = analysis_df['next_arrive_time'].quantile(0.99)
time_q99

np.float64(150.0)

In [48]:
long_time_df = analysis_df[analysis_df['next_arrive_time'] >= time_q99]

long_time_df[[
    'route_nm',
    'now_station',
    'next_station',
    'distance',
    'now_arrive_time',
    'next_arrive_time'
]].sort_values('next_arrive_time', ascending = False).head(20)

,route_nm,now_station,next_station,distance,now_arrive_time,next_arrive_time
208573,281-1,동문로터리,주공 3 4단지,549.0,10시,154
208277,281-1,제주대학교병원,죽성마을 입구,738.0,06시,154
208027,281-1,비석거리,세기아파트,237.0,09시,154
207894,281-1,제주대학교병원,죽성마을 입구,738.0,09시,154
207763,281-1,입석동,하례환승정류장(하례리입구),1949.0,09시,154
207541,281-1,제주대학교병원,죽성마을 입구,738.0,14시,154
207471,281-1,서귀포여자고등학교,LH아파트,858.0,20시,154
207427,281-1,비석거리,세기아파트,237.0,16시,154
207398,281-1,제주시청(아라방면),고산동산(아라방면),454.0,15시,154
207386,281-1,동문로터리,주공 3 4단지,549.0,12시,154


In [50]:
# 다음 정거장 기타 열 생성 기준(기준: 빈도 두 많은 범주, 상위 12개)
q90_01 = df['next_arrive_time'].quantile(0.9)
df[df['next_arrive_time']>q90_01]['now_station'].value_counts().head(50)

now_station
고산동산(아라방면)         883
제원아파트              755
제주여자중고등학교(광양방면)    732
제주버스터미널            538
삼무공원사거리            517
한라병원               485
남녕고등학교             467
인다마을               452
한국병원               450
청소년문화의집            435
시민회관               413
아라주공아파트            382
도호동                357
아라동주민센터            328
주공 3 4단지           297
아라초등학교             295
한라중학교/부영아파트        289
서귀포시 구 버스터미널       265
제주여자중고등학교(아라방면)    253
남국사                249
용담1동주민센터           249
삼양2동               247
제주시청(광양방면)         239
제주시청(아라방면)         220
은남동                218
보성시장               195
연동입구               194
동성마을               192
서문시장               192
연동대림1차아파트          191
인화초등학교             187
용문마을회관             185
하례환승정류장(하례리입구)     179
한동리서동              176
종달초등학교             174
관덕정                172
월성 마을              171
중앙로(국민은행)          165
탐라장애인 종합복지관        156
입석동                155
동광양                152
고성리제주은행            145
평생학습관              139